In [ ]:
"""
Сейчас сентябрь 2024 года. 
В каждом магазине сети есть цех по производству готовых блюд. 
Проведите ABC-анализ (по марже) за сентябрь товаров собственного производства в магазине, имеющем самые большие продажи, 
найдите среди товаров категории А товары, по которым в сентябре ухудшилась оборачиваемость по отношению к прошлому месяцу, 
и выберите среди них 5 товаров с самой высокой рентабельностью продаж (за 3 месяца). 
"""


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

def analiz_unique(t):
    cols=list(t.columns)
    return display([(c,t[c].unique()) for c in cols])

convert={'Сумма продаж, тыс.руб.': lambda x: float(x.replace(',','.')),'Себестоимость продаж, тыс.руб.':lambda x: float(x.replace(',','.')),'Себестоимость остатков на конец дня среднедневная, тыс.руб.':lambda x: float(x.replace(',','.'))}

### Разведочный анализ
df=pd.read_csv('_Тестовое задание 1 (выгрузка).csv', encoding='cp1251', sep='\t', converters=convert)

df.head()

,Формат магазина,Магазин,Группа,Номенклатура,Месяц,"Кол проданного товара, шт","Сумма продаж, тыс.руб.","Себестоимость продаж, тыс.руб.","Остаток на конец дня среднедневной, шт","Себестоимость остатков на конец дня среднедневная, тыс.руб."
0,Гипер,Магазин 3,Молочные продукты,Молочные продукты 1,7,19,0.81251,0.66358,5,0.17462
1,Гипер,Магазин 3,Молочные продукты,Молочные продукты 1,8,20,0.88241,0.69850,5,0.17462
2,Гипер,Магазин 3,Молочные продукты,Молочные продукты 1,9,40,1.60159,1.18046,5,0.13970
3,Гипер,Магазин 3,Молочные продукты,Молочные продукты 2,7,70,2.41666,1.84436,8,0.24270
4,Гипер,Магазин 3,Молочные продукты,Молочные продукты 2,8,55,1.87875,1.43176,9,0.21839


In [29]:
### Анализ Магазинов
gr_magaz=df.groupby(['Магазин', 'Месяц'], as_index=False).agg({'Сумма продаж, тыс.руб.':'sum'}).sort_values('Сумма продаж, тыс.руб.', ascending=False)
display(gr_magaz.nlargest(5, 'Сумма продаж, тыс.руб.'))
px.line(gr_magaz.astype({'Месяц':str}), x='Месяц', y='Сумма продаж, тыс.руб.', color='Магазин', title='Продажи в магазинах',width=1000)

,Магазин,Месяц,"Сумма продаж, тыс.руб."
0,Магазин 11,7,12683.65663
1,Магазин 11,8,11779.31358
2,Магазин 11,9,11243.55773
33,Магазин 8,7,8806.95344
34,Магазин 8,8,8140.58768


**Вывод** 
+ Для анализа оборачиваемости выбираем магазин с самымм большими продажами - Магазин 11 

In [3]:
### Датасет для анализа
df_9_11=df.query('Месяц==9&Группа=="Производство"&Магазин=="Магазин 11"')
df_9_11['Маржа']=df_9_11['Сумма продаж, тыс.руб.']-df_9_11['Себестоимость продаж, тыс.руб.']

### Abc анализ
df_9_gr=df_9_11.groupby('Номенклатура', as_index=False)['Маржа'].sum().sort_values('Маржа', ascending=False).assign(Кумул=lambda x: x['Маржа'].cumsum()).reset_index(drop=True)
df_9_gr['Процент']=df_9_gr['Кумул']/df_9_gr['Маржа'].sum()
cond=[df_9_gr['Процент']<=0.8,(0.8<df_9_gr['Процент'])&(df_9_gr['Процент']<=0.95),df_9_gr['Процент']>0.95]
df_9_gr['type']=np.select(cond, ['A','B','C'])
# df_9_gr.to_csv('2.abc_analiz.csv', index=False)
display(df_9_gr.value_counts('type', normalize=True))
df_9_gr.head() 

C:\Users\Рин\AppData\Local\Temp\ipykernel_3880\3924001978.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



type
C    0.397260
A    0.328767
B    0.273973
Name: proportion, dtype: float64

,Номенклатура,Маржа,Кумул,Процент,type
0,Производство 13,8.61598,8.61598,0.126609,A
1,Производство 29,6.10186,14.71784,0.216274,A
2,Производство 105,5.46879,20.18663,0.296636,A
3,Производство 5,4.88405,25.07068,0.368406,A
4,Производство 86,3.61241,28.68309,0.421489,A


In [28]:
# категории А товары, по которым в сентябре ухудшилась оборачиваемость по отношению к прошлому месяцу
a_tovs=df_9_gr.query('type=="A"')['Номенклатура'].to_list()
df_tov_a=df[df['Номенклатура'].isin(a_tovs)& (df['Магазин']=='Магазин 11')].sort_values('Номенклатура', ascending=False)#.to_csv('table_tovs_a.csv', index=False)
# df_tov_a['дней_месяц']=np.select([df_tov_a['Месяц']==9,df_tov_a['Месяц']==8,df_tov_a['Месяц']==7],[30,31,31])

df_piv_a=df_tov_a.pivot(index=['Магазин','Номенклатура'], columns=['Месяц'], values=['Себестоимость продаж, тыс.руб.','Себестоимость остатков на конец дня среднедневная, тыс.руб.']).reset_index()
df_piv_a['Оборачиваемость_7']=df_piv_a[('Себестоимость остатков на конец дня среднедневная, тыс.руб.',7)]*31/df_piv_a[('Себестоимость продаж, тыс.руб.',7)]
df_piv_a['Оборачиваемость_8']=df_piv_a[('Себестоимость остатков на конец дня среднедневная, тыс.руб.',8)]*31/df_piv_a[('Себестоимость продаж, тыс.руб.',8)]
df_piv_a['Оборачиваемость_9']=df_piv_a[('Себестоимость остатков на конец дня среднедневная, тыс.руб.',9)]*31/df_piv_a[('Себестоимость продаж, тыс.руб.',9)]
df_piv_a['Изм_9_8']=df_piv_a['Оборачиваемость_9']/df_piv_a['Оборачиваемость_8']-1
df_piv_a.to_csv('3.oborachivaemost.csv', index=False)
print('='*50)
print('Товары, у которых ухудшилась оборачиваемость м/м\n')
display(df_piv_a[df_piv_a[('Изм_9_8','')]>0])

fig_1=px.bar(df_piv_a, x=df_piv_a[('Номенклатура', '')], y=df_piv_a[('Изм_9_8','')], title='Оборачиваемости  09.24 к 08,24', width=1000)
fig_1.update_layout(xaxis_title='Номенклатура', yaxis_title='Изм-е')
fig_1.show()

Товары, у которых ухудшилась оборачиваемость м/м



Магазин     Номенклатура Себестоимость продаж, тыс.руб.           \
Месяц                                                           7        8   
1      Магазин 11  Производство 11                        1.03619  1.48995   
6      Магазин 11  Производство 38                        3.81120  2.58050   
12     Магазин 11  Производство 45                        2.33700  1.72200   
16     Магазин 11  Производство 66                        1.48446  1.87290   
18     Магазин 11  Производство 75                        1.79376  1.58064   
20     Магазин 11  Производство 80                        2.71810  2.92990   
21     Магазин 11  Производство 83                        1.02760  1.08632   

               Себестоимость остатков на конец дня среднедневная, тыс.руб.  \
Месяц        9                                                           7   
1      1.94040                                            1.66322            
6      3.22620                                            0.15880            
12     2.33700                                            0.18450            
16     1.66480                                            0.29134            
18     1.63392                                            0.05328            
20     2.64750                                            0.03530            
21     1.79830                                            0.20552            

                        Оборачиваемость_7 Оборачиваемость_8 Оборачиваемость_9  \
Месяц        8        9                                                         
1      0.97020  1.48995         49.759040         20.186047         23.803571   
6      0.27790  0.43200          1.291667          3.338462          4.151014   
12     0.14350  0.26650          2.447368          2.583333          3.535088   
16     0.41620  0.45782          6.084058          6.888889          8.525000   
18     0.15984  0.23088          0.920792          3.134831          4.380435   
20     0.28240  0.28240          0.402597          2.987952          3.306667   
21     0.17616  0.40370          6.200000          5.027027          6.959184   

        Изм_9_8  
Месяц            
1      0.179209  
6      0.243391  
12     0.368421  
16     0.237500  
18     0.397343  
20     0.106667  
21     0.384354

In [9]:
## выберите среди них 5 товаров с самой высокой рентабельностью продаж (за 3 месяца)
a_tovs_less=df_piv_a[df_piv_a[("Изм_9_8","")]>0]['Номенклатура'].to_list()
print(len(a_tovs_less))
df_tov_a_less=df.query('Номенклатура.isin(@a_tovs_less)&Магазин=="Магазин 11"')
margin_a_less=df_tov_a_less.groupby('Номенклатура', as_index=False).agg({'Сумма продаж, тыс.руб.':'sum','Себестоимость продаж, тыс.руб.':'sum'}).assign(margin=lambda x: (x['Сумма продаж, тыс.руб.']-x['Себестоимость продаж, тыс.руб.'])/x['Сумма продаж, тыс.руб.']).sort_values('margin', ascending=False)
margin_a_less.to_csv('4.margin.csv', index=False)
margin_a_less[:5]

7


,Номенклатура,"Сумма продаж, тыс.руб.","Себестоимость продаж, тыс.руб.",margin
5,Производство 80,14.71055,8.29550,0.436085
6,Производство 83,6.79061,3.91222,0.423878
3,Производство 66,8.18551,5.02216,0.386457
1,Производство 38,14.69576,9.61790,0.345532
4,Производство 75,7.38745,5.00832,0.322050


In [ ]:
"""
На основе данных из предыдущего задания выявите главную (на ваш взгляд) проблему указанной торговой сети 
в части собственного производства готовых блюд (обоснуйте, почему). 
Что Вы предложите для решения этой проблемы и какой эффект могло бы принести Ваше решение? 
Чтобы долго не писать, запишите голосовое сообщение и приложите ссылку на него.
"""

In [27]:
total_grup=df.groupby(['Месяц','Группа'], as_index=False).agg({'Сумма продаж, тыс.руб.':'sum','Себестоимость продаж, тыс.руб.':'sum'}).assign(Маржа=lambda x: x['Сумма продаж, тыс.руб.']-x['Себестоимость продаж, тыс.руб.']).assign(Маржинальность=lambda x: x['Маржа']/x['Сумма продаж, тыс.руб.'])
total_grup['dolya']=total_grup['Сумма продаж, тыс.руб.']/total_grup.groupby('Месяц')['Сумма продаж, тыс.руб.'].transform('sum')
total_grup['izm_mm']=total_grup.sort_values(['Месяц']).groupby('Группа')['Сумма продаж, тыс.руб.'].pct_change()
print('='*50, "Сегмент - Группы\n")
display(total_grup)
metrics_t=['Сумма продаж, тыс.руб.','Маржа', 'Маржинальность']
for m in metrics_t:
    fig_tot=px.line(total_grup.astype({'Месяц':str}), x='Месяц',y=m, color='Группа', title=f'{m}_групп', width=1000)
    fig_tot.show()



proiz=df.query('Группа=="Производство"').assign(Маржа=lambda x: x['Сумма продаж, тыс.руб.']-x['Себестоимость продаж, тыс.руб.'])
metrics=['Сумма продаж, тыс.руб.','Маржа']

def agg_cat(t, cat):

    agg_t=t.groupby(['Месяц',cat], as_index=False).agg({'Сумма продаж, тыс.руб.':'sum','Маржа':'sum'})
    agg_t['Изменения продаж м/м']=agg_t.groupby(cat)['Сумма продаж, тыс.руб.'].pct_change()
    for m in metrics:
        fig=px.line(agg_t.astype({'Месяц':str}), x='Месяц', y=m, color=cat, title=f'{m}_Производство_{cat}',width=1000)
        fig.show()
    fig_1=px.bar(agg_t.astype({'Месяц':str}), x='Месяц', y='Изменения продаж м/м', color=cat, barmode='group', title=f'Изменение продаж м/м - {cat}',width=1000)
    fig_1.show()

    return agg_t

print('='*50,"Сегмент - Формат магазина\n")
display(agg_cat(proiz,'Формат магазина'))
print('='*50, "Сегмент - Магазины\n")
display(agg_cat(proiz,'Магазин'))



================================================== Сегмент - Группы



,Месяц,Группа,"Сумма продаж, тыс.руб.","Себестоимость продаж, тыс.руб.",Маржа,Маржинальность,dolya,izm_mm
0,7,Молочные продукты,22766.36918,17261.73872,5504.63046,0.241788,0.349951,NaN
1,7,Мясо и мясопродукты,21075.28915,16288.09743,4787.19172,0.227147,0.323957,NaN
2,7,Овощи и фрукты,13637.97383,10891.42173,2746.55210,0.201390,0.209635,NaN
3,7,Производство,682.79166,486.91999,195.87167,0.286869,0.010495,NaN
4,7,Хлеб и хлебобулочные изд,6893.40011,5015.44915,1877.95096,0.272427,0.105961,NaN
5,8,Молочные продукты,21886.52661,16445.08673,5441.43988,0.248621,0.362389,-0.038647
6,8,Мясо и мясопродукты,20409.55690,15721.52486,4688.03204,0.229698,0.337934,-0.031588
7,8,Овощи и фрукты,10490.92551,8375.72439,2115.20112,0.201622,0.173705,-0.230756
8,8,Производство,687.98105,469.76721,218.21384,0.317180,0.011391,0.007600
9,8,Хлеб и хлебобулочные изд,6920.05235,4998.56930,1921.48305,0.277669,0.114580,0.003866


================================================== Сегмент - Формат магазина



,Месяц,Формат магазина,"Сумма продаж, тыс.руб.",Маржа,Изменения продаж м/м
0,7,Гипер,479.61461,128.70157,NaN
1,7,Супер,203.17705,67.17010,NaN
2,8,Гипер,470.75112,136.58916,-0.018480
3,8,Супер,217.22993,81.62468,0.069166
4,9,Гипер,491.78402,150.30539,0.044679
5,9,Супер,265.42484,106.80716,0.221861


================================================== Сегмент - Магазины



,Месяц,Магазин,"Сумма продаж, тыс.руб.",Маржа,Изменения продаж м/м
0,7,Магазин 11,242.94362,57.60442,NaN
1,7,Магазин 14,36.58616,10.95244,NaN
2,7,Магазин 16,24.63457,8.22298,NaN
3,7,Магазин 20,23.63794,8.42144,NaN
4,7,Магазин 22,30.95916,9.11967,NaN
5,7,Магазин 25,21.68760,8.06614,NaN
6,7,Магазин 29,23.57092,7.93586,NaN
7,7,Магазин 3,50.38830,17.06402,NaN
8,7,Магазин 30,23.37255,7.60996,NaN
9,7,Магазин 31,18.72815,6.84161,NaN


**Вывод**
+ Доля "Производства" в общей выручке около 1%
+ При этом она одна из немногих категорий которая растет по продажам. За 2 месяца показала рост на 11%, это рекорд среди всех категорий. 
+ Для увеличения продаж по данной категории можно поработать с оборачиваемостью номенклатуры (смю анализ оборачиваемости) а так же с магазинами, в которых падают продажи. В особенности с магазином 4 (изначально он второй по объему продаж)
+ Так же стоит поработать с подкатегориями товаров внутри данной категории